# Text Cleaning: Repairing Words Joined When the HTML Was Removed

When the postings were collected, their HTML was stripped without inserting spaces, so list items and paragraphs run together: *"platformDevelop"*, *"them.Carefully"*, *"InstituteOur"*. The word TF-IDF of the selected Step 2 model treats each of these as a single invented token ("platformdevelop"), so the words on both sides are lost. HTML entities such as `&amp;` also remain in the text. The character n-grams of the model can partly recover the split words, which may explain why the model works well despite this.

This notebook measures how common the problem is and tests whether repairing the text improves the selected model. The test set is not used.

### Change and decision rule (fixed before running)

`clean_posting_text()` (in `fakejob/features.py`) changes each posting on its own, so nothing is learned across rows:

1. HTML entities are decoded (`&amp;` → `&`).
2. A space is inserted between a lower-case letter and a following capitalised word (`platformDevelop` → `platform Develop`).
3. A space is inserted after sentence punctuation directly followed by a letter (`them.Carefully` → `them. Carefully`).
4. Runs of whitespace are collapsed.

Rule 2 also splits a few genuine names (`JavaScript` → `Java Script`); this is accepted because it is applied the same way to every posting. The similarity groups and the split are built from the original text, so they do not change.

**Candidate:** `STEP2_CONFIG` on the cleaned text. **Baseline:** the saved Step 2 results on the original text (`results/step2_combined_folds.csv`), which use the same folds (5×3, seed 0).

**Decision rule.** The aim is F1, so the rule of `f1_improvements.ipynb` is used: cleaning is adopted when the mean paired F1 gain is positive **and** it is better in at least 12 of the 15 folds. PR-AUC is reported alongside. No other cleaning variants are added after seeing the results.

In [1]:
import sklearn

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        f"This notebook uses scikit-learn 1.6.1 (found {sklearn.__version__}). "
        "Install the pinned versions with: pip install -r requirements.txt"
    )

import re
import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from pathlib import Path

import numpy as np
import pandas as pd

from fakejob.data import load_split
from fakejob.evaluation import CVResult, compare, passes_rule, repeated_group_cv
from fakejob.features import STEP2_CONFIG, build_pipeline, clean_posting_text, posting_features

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 90)
RESULTS_DIR = Path("results")
METRICS = ["average_precision", "roc_auc", "f1", "precision", "recall"]
TEXT_FIELDS = ["title", "company_profile", "description", "requirements", "benefits"]

split = load_split()
dev_index = split.dev_index
dev = split.model_df.loc[dev_index]
y_dev = split.y.loc[dev_index]
groups_dev = split.groups.loc[dev_index]
print(f"Development data: {len(dev_index)} postings, {int(y_dev.sum())} fake")

Development data: 15001 postings, 724 fake


## How Common Is the Problem?

Counts on the development postings. A *joined word* is a lower-case letter followed (optionally after `.`, `!` or `?`) by a capitalised word.

In [2]:
JOINED = r"[a-z][.!?]?[A-Z][a-z]"
rows = []
for field in TEXT_FIELDS:
    original = dev[field].fillna("")
    cleaned = clean_posting_text(original)
    filled = original.str.strip() != ""
    rows.append({
        "field": field,
        "postings with text": int(filled.sum()),
        "with joined words (%)": 100 * (original.str.count(JOINED) > 0)[filled].mean(),
        "joined words per posting (median)": original.str.count(JOINED)[filled].median(),
        "with HTML entities (%)": 100 * original.str.contains(r"&(?:amp|nbsp|lt|gt|quot|#\d+);")[filled].mean(),
        "joined words left after cleaning (%)": 100 * (cleaned.str.count(JOINED) > 0)[filled].mean(),
    })
display(pd.DataFrame(rows).set_index("field").round(1))

by_label = pd.DataFrame({
    "label": y_dev.map({0: "real", 1: "fake"}),
    "joined": dev["description"].fillna("").str.count(JOINED),
}).groupby("label")["joined"].agg(["mean", "median"])
print("Joined words in the description by label:")
display(by_label.round(2))

,postings with text,with joined words (%),joined words per posting (median),with HTML entities (%),joined words left after cleaning (%)
field,,,,,
title,15001,2.0,0.0,0.0,0.0
company_profile,12203,58.6,1.0,19.8,0.0
description,15000,83.9,4.0,21.6,0.0
requirements,12797,82.7,4.0,14.3,0.0
benefits,9026,57.2,1.0,12.4,0.0


Joined words in the description by label:


,mean,median
label,,
fake,4.39,2.0
real,5.84,4.0


### Examples

Five development descriptions: the same span of text around the first joined word, before and after cleaning.

In [3]:
def excerpt(text, width=120):
    match = re.search(JOINED, text)
    start = max(0, match.start() - 50)
    return text[start:start + width]

sample = dev["description"].fillna("")
sample = sample[sample.str.count(JOINED) > 0].sample(5, random_state=0)
before = sample.map(excerpt)
display(pd.DataFrame({"before": before, "after": clean_posting_text(before)}).reset_index(drop=True))

,before,after
0,"Position: BIG DATA - Hadoop DesignerLocation: Los Angeles, CA - 00592Duration: 6 month...","Position: BIG DATA - Hadoop Designer Location: Los Angeles, CA - 00592Duration: 6 mont..."
1,and guide them through their growth and development.You might have heard of some of th...,and guide them through their growth and development. You might have heard of some of t...
2,uitment and then be kept on in a permanent position.You will be working for a recruitm...,uitment and then be kept on in a permanent position. You will be working for a recruit...
3,"nal &amp; strategic people strategy of the business.Leading a small team, you will be ...","nal & strategic people strategy of the business. Leading a small team, you will be rel..."
4,nd experience working on software development teams.You will define and lead our passi...,nd experience working on software development teams. You will define and lead our pass...


## Effect on the Vocabulary

Word unigrams and bigrams as in `STEP2_CONFIG` (`min_df = 2`), fitted on all development postings only to describe the vocabulary; the model itself fits its vocabulary inside each training fold. An *invented* token is a word that contains a lower-case to upper-case join in the original text.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

def combined(frame, clean):
    feats = posting_features(frame, clean_text=clean)
    return feats["combined_text"]

joined_tokens = set()
for text in dev["description"].fillna(""):
    joined_tokens.update(m.lower() for m in re.findall(r"\b[A-Za-z]*[a-z][A-Z][a-z]+\b", text))

rows = []
for name, clean in [("original", False), ("cleaned", True)]:
    vocab = TfidfVectorizer(ngram_range=(1, 1), min_df=2).fit(combined(dev, clean)).vocabulary_
    rows.append({"text": name, "word types (min_df = 2)": len(vocab),
                 "invented joined tokens in vocabulary": len(joined_tokens & set(vocab))})
display(pd.DataFrame(rows).set_index("text"))

,word types (min_df = 2),invented joined tokens in vocabulary
text,,
original,29603,6074
cleaned,23690,227


## Cross-Validation

The candidate is run with the project protocol. The baseline is loaded from the saved Step 2 results; the assertion checks that the folds match.

In [5]:
baseline_folds = pd.read_csv(RESULTS_DIR / "step2_combined_folds.csv")
baseline = CVResult(name="original text", folds=baseline_folds, oof_scores=None)

start = time.time()
X_clean = posting_features(split.model_df, clean_text=True).loc[dev_index]
cleaned = repeated_group_cv(
    build_pipeline(STEP2_CONFIG), X_clean, y_dev, groups_dev, name="cleaned text",
    n_splits=5, n_repeats=3, inner_splits=3, seed=0, n_jobs=4,
)
cleaned.folds.to_csv(RESULTS_DIR / "text_cleaning_folds.csv", index=False)
print(f"cleaned text: {time.time() - start:.0f}s")

assert (cleaned.folds[["repeat", "fold", "n_test", "n_fake"]].values
        == baseline.folds[["repeat", "fold", "n_test", "n_fake"]].values).all()

comparison = compare(baseline, cleaned, metrics=METRICS)
display(comparison)
print("Passes F1 rule:", passes_rule(comparison, metric="f1"))
print("Would pass PR-AUC rule:", passes_rule(comparison, metric="average_precision"))

cleaned text: 1235s


,baseline,candidate,mean_diff,std_diff,folds_improved
metric,,,,,
average_precision,0.871,0.868,-2.395e-03,3.517e-03,5/15
roc_auc,0.984,0.984,-3.920e-04,5.874e-04,3/15
f1,0.789,0.792,2.324e-03,1.922e-02,9/15
precision,0.834,0.839,4.958e-03,5.505e-02,6/15
recall,0.752,0.754,1.835e-03,2.993e-02,6/15


Passes F1 rule: False
Would pass PR-AUC rule: False


### Per Repeat

In [6]:
per_repeat = pd.concat({
    "original": baseline.folds.groupby("repeat")[["average_precision", "f1", "precision", "recall"]].mean(),
    "cleaned": cleaned.folds.groupby("repeat")[["average_precision", "f1", "precision", "recall"]].mean(),
}, axis=1)
per_repeat

original                                   cleaned         \
       average_precision     f1 precision recall average_precision     f1   
repeat                                                                      
0                  0.885  0.801     0.857  0.751             0.883  0.795   
1                  0.859  0.779     0.801  0.760             0.856  0.779   
2                  0.867  0.789     0.843  0.744             0.866  0.801   

                         
       precision recall  
repeat                   
0          0.838  0.761  
1          0.823  0.741  
2          0.856  0.760

## Findings

**Cleaning does not pass the rule, so the selected model keeps the original text.**

| | Original text | Cleaned text | Mean paired gain | Folds better |
|---|---|---|---|---|
| F1 | 0.789 | 0.792 | +0.002 | 9/15 |
| Precision | 0.834 | 0.839 | +0.005 | 6/15 |
| Recall | 0.752 | 0.754 | +0.002 | 6/15 |
| PR-AUC | 0.871 | 0.868 | −0.002 | 5/15 |

- **The problem is real and common.** Joined words occur in 84% of descriptions and 83% of requirements (median 4 per posting), and HTML entities in about a fifth of them. Cleaning removes all of them and shrinks the word vocabulary (`min_df = 2`) from 29,603 to 23,690 types; 6,074 invented joined tokens disappear.
- **The model was not harmed by it.** The F1 gain is small and inconsistent (9 of 15 folds, standard deviation 0.019, ten times the mean), and PR-AUC is slightly lower in 10 of 15 folds. Two reasons are likely. The character n-grams (`char_wb`, 3–5) already see the pieces of a joined word, so the information was not lost. And the joins themselves carry a small signal: they come from HTML lists and paragraphs, and real postings contain more of them than fake ones (mean 5.8 against 4.4 per description), which cleaning removes.
- **Consequence for Step 5.** The DistilBERT notebook keeps the original text, as fixed in its design. Its WordPiece tokenizer also splits joined words into sub-word pieces, so the issue matters less there.

The result is recorded as a data-quality check: the text has a common formatting defect, but repairing it does not change the selected model. The test set was not used.